# Train YOLO26s-seg for paper counting on Kaggle

This notebook trains a dedicated one-class **instance segmentation** model (`paper`). Keep `best (1).pt` for smartphone/earphone/smartwatch and use this checkpoint only in person/desk/lap ROIs.

Attach a custom dataset exported as **YOLOv8 Segmentation**. Every physical sheet, including the authorized exam paper, must have its own polygon instance. Optionally attach `ahmedezzat02/datazeft`; only valid `Book` polygons are reused. Detection boxes are deliberately not converted into fake masks.

In [ ]:
%pip install -q -U "ultralytics>=8.4.0,<9" pyyaml pandas matplotlib


In [ ]:
from __future__ import annotations
import hashlib, json, os, random, shutil
from collections import Counter, defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import torch, yaml
from IPython.display import FileLink, display
from PIL import Image, ImageDraw
from ultralytics import YOLO, __version__ as ultralytics_version

SEED = 42
random.seed(SEED)
print('Ultralytics:', ultralytics_version)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator in Kaggle Notebook options.')
print('GPU:', torch.cuda.get_device_name(0))


## 1. Locate the attached datasets

If auto-selection is ambiguous, set `CUSTOM_DATASET_ROOT` to one of the paths printed by this cell.

In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/paper_yolo26s_seg')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def dataset_roots():
    roots = []
    for yaml_path in KAGGLE_INPUT.rglob('data.yaml'):
        root = yaml_path.parent
        if (root / 'train' / 'images').is_dir():
            roots.append(root)
    return sorted(set(roots), key=str)

ROOTS = dataset_roots()
print('Attached YOLO datasets:')
for index, root in enumerate(ROOTS):
    print(index, root)

CUSTOM_DATASET_ROOT = None  # Example: Path('/kaggle/input/paper-lap-seg')
if CUSTOM_DATASET_ROOT is None:
    candidates = [root for root in ROOTS if 'datazeft' not in str(root).lower() and any(token in str(root).lower() for token in ('paper', 'sheet', 'lap'))]
    if len(candidates) != 1:
        raise RuntimeError('Set CUSTOM_DATASET_ROOT manually to the custom paper segmentation dataset printed above.')
    CUSTOM_DATASET_ROOT = candidates[0]
CUSTOM_DATASET_ROOT = Path(CUSTOM_DATASET_ROOT)
PUBLIC_DATAZEFT_ROOT = next((root for root in ROOTS if 'datazeft' in str(root).lower()), None)
print('Custom dataset:', CUSTOM_DATASET_ROOT)
print('Optional public data:', PUBLIC_DATAZEFT_ROOT)


## 2. Build a clean one-class polygon dataset

The custom dataset may contain empty-label negative images. It must not contain paper rows with only four bbox coordinates.

In [ ]:
OUTPUT_DATASET = WORK_ROOT / 'dataset'
SPLIT_FOLDERS = {'train': 'train', 'val': 'valid', 'test': 'test'}
for split in SPLIT_FOLDERS:
    (OUTPUT_DATASET / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUTPUT_DATASET / split / 'labels').mkdir(parents=True, exist_ok=True)

def yaml_names(root):
    payload = yaml.safe_load((root / 'data.yaml').read_text(encoding='utf-8'))
    raw = payload.get('names', {})
    if isinstance(raw, list):
        return {index: str(name) for index, name in enumerate(raw)}
    return {int(index): str(name) for index, name in raw.items()}

def normalized_name(value):
    return value.lower().strip().replace('-', '_').replace(' ', '_')

PAPER_ALIASES = {'paper', 'book', 'cheat_sheet', 'cheatsheet', 'test_paper', 'exam_paper'}
stats = Counter()
groups_by_split = defaultdict(set)

def add_source(root, source_tag, include_empty):
    names = yaml_names(root)
    paper_ids = {index for index, name in names.items() if normalized_name(name) in PAPER_ALIASES}
    if not paper_ids:
        raise RuntimeError(f'No paper-like class found in {root}: {names}')
    for split, folder in SPLIT_FOLDERS.items():
        image_dir = root / folder / 'images'
        label_dir = root / folder / 'labels'
        if not image_dir.is_dir():
            raise FileNotFoundError(image_dir)
        for image_path in sorted(image_dir.rglob('*')):
            if image_path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            label_path = label_dir / f'{image_path.stem}.txt'
            raw_lines = label_path.read_text(encoding='utf-8').splitlines() if label_path.is_file() else []
            polygons = []
            selected_box_rows = 0
            for raw_line in raw_lines:
                parts = raw_line.split()
                if len(parts) < 5:
                    stats['invalid_rows'] += 1
                    continue
                try:
                    class_id = int(float(parts[0]))
                    coordinates = [float(value) for value in parts[1:]]
                except ValueError:
                    stats['invalid_rows'] += 1
                    continue
                if class_id not in paper_ids:
                    continue
                if len(coordinates) == 4:
                    selected_box_rows += 1
                    continue
                valid = len(coordinates) >= 6 and len(coordinates) % 2 == 0 and all(0 <= value <= 1 for value in coordinates)
                if not valid:
                    stats['invalid_rows'] += 1
                    continue
                polygons.append('0 ' + ' '.join(f'{value:.8f}' for value in coordinates))
            stats[f'{source_tag}_box_rows_skipped'] += selected_box_rows
            if source_tag == 'custom' and selected_box_rows:
                stats['custom_images_with_detection_boxes'] += 1
            if not polygons and not include_empty:
                continue
            digest = hashlib.sha1(str(image_path).encode('utf-8')).hexdigest()[:10]
            target_stem = f'{source_tag}_{digest}_{image_path.stem}'
            target_image = OUTPUT_DATASET / split / 'images' / f'{target_stem}{image_path.suffix.lower()}'
            target_label = OUTPUT_DATASET / split / 'labels' / f'{target_stem}.txt'
            if not target_image.exists():
                try:
                    os.symlink(image_path, target_image)
                except OSError:
                    shutil.copy2(image_path, target_image)
            target_label.write_text(('\n'.join(polygons) + '\n') if polygons else '', encoding='utf-8')
            stats[f'{source_tag}_{split}_images'] += 1
            stats[f'{source_tag}_{split}_instances'] += len(polygons)
            group = image_path.name.split('.rf.', 1)[0]
            groups_by_split[f'{source_tag}:{group}'].add(split)

add_source(CUSTOM_DATASET_ROOT, 'custom', include_empty=True)
if PUBLIC_DATAZEFT_ROOT is not None:
    add_source(PUBLIC_DATAZEFT_ROOT, 'public', include_empty=False)

custom_instances = sum(stats[f'custom_{split}_instances'] for split in SPLIT_FOLDERS)
assert custom_instances > 0, 'No custom paper polygons found. Export YOLO segmentation, not detection.'
assert stats['custom_images_with_detection_boxes'] == 0, 'Custom paper labels contain detection boxes. Re-export YOLOv8 Segmentation.'
for split in SPLIT_FOLDERS:
    assert any((OUTPUT_DATASET / split / 'images').iterdir()), f'No images were prepared for split: {split}'
leaked = {group: sorted(splits) for group, splits in groups_by_split.items() if len(splits) > 1}
display(pd.Series(stats).sort_index().to_frame('count'))
print('Cross-split source groups:', len(leaked))
if leaked:
    print('WARNING: split adjacent/augmented frames by source video before trusting metrics.')
    print(list(leaked.items())[:10])

DATA_YAML = WORK_ROOT / 'paper_seg.yaml'
DATA_YAML.write_text(yaml.safe_dump({'path': str(OUTPUT_DATASET), 'train': 'train/images', 'val': 'val/images', 'test': 'test/images', 'names': {0: 'paper'}}, sort_keys=False), encoding='utf-8')
print(DATA_YAML.read_text(encoding='utf-8'))


## 3. Visual polygon audit

Stop if two touching sheets are merged into one polygon. The model cannot learn an instance boundary missing from the labels.

In [ ]:
def list_images(directory):
    return sorted(path for path in directory.iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)

def draw_polygons(image_path):
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    width, height = image.size
    label_path = image_path.parent.parent / 'labels' / f'{image_path.stem}.txt'
    for raw_line in label_path.read_text(encoding='utf-8').splitlines():
        coordinates = [float(value) for value in raw_line.split()[1:]]
        points = [(coordinates[index] * width, coordinates[index + 1] * height) for index in range(0, len(coordinates), 2)]
        draw.line(points + [points[0]], fill='red', width=4)
    return image

train_images = list_images(OUTPUT_DATASET / 'train' / 'images')
samples = random.sample(train_images, min(12, len(train_images)))
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for axis in axes.flat:
    axis.axis('off')
for axis, image_path in zip(axes.flat, samples):
    axis.imshow(draw_polygons(image_path))
    axis.set_title(image_path.name[:45], fontsize=8)
plt.tight_layout()


## 4. Train YOLO26s-seg

`s` is the recommended quality/speed balance for Kaggle T4/P100. Use `yolo26n-seg.pt` only if CPU latency matters more than recall.

In [ ]:
SMOKE_TEST = False
MODEL_NAME = 'yolo26s-seg.pt'
RUN_NAME = 'paper_yolo26s_seg_768'
EPOCHS = 2 if SMOKE_TEST else 120
FRACTION = 0.05 if SMOKE_TEST else 1.0
model = YOLO(MODEL_NAME)
model.info()
model.train(
    data=str(DATA_YAML), epochs=EPOCHS, patience=25, imgsz=768, batch=8,
    device=0, workers=4, project=str(WORK_ROOT), name=RUN_NAME, exist_ok=True,
    pretrained=True, optimizer='auto', cos_lr=True, close_mosaic=15, amp=True,
    seed=SEED, deterministic=True, cache=False, fraction=FRACTION, plots=True,
    save=True, save_period=10, overlap_mask=True, mask_ratio=4,
    hsv_h=0.015, hsv_s=0.45, hsv_v=0.35, degrees=8.0, translate=0.10,
    scale=0.40, shear=2.0, perspective=0.0005, fliplr=0.50, mosaic=0.50,
    mixup=0.0, copy_paste=0.15,
)
RUN_DIR = WORK_ROOT / RUN_NAME
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
LAST_PT = RUN_DIR / 'weights' / 'last.pt'
assert BEST_PT.is_file(), f'best.pt was not created: {BEST_PT}'
print('Best checkpoint:', BEST_PT)

# To resume an interrupted run in the same Kaggle session:
# YOLO(str(LAST_PT)).train(resume=True)


## 5. Evaluate both boxes and masks on held-out clips

In [ ]:
best_model = YOLO(str(BEST_PT))
metrics = best_model.val(data=str(DATA_YAML), split='test', imgsz=960, batch=8, device=0, conf=0.001, iou=0.60, plots=True, project=str(WORK_ROOT), name='paper_test_960')
summary = pd.DataFrame([{
    'box_mAP50': metrics.box.map50, 'box_mAP50-95': metrics.box.map,
    'box_precision': metrics.box.mp, 'box_recall': metrics.box.mr,
    'mask_mAP50': metrics.seg.map50, 'mask_mAP50-95': metrics.seg.map,
    'mask_precision': metrics.seg.mp, 'mask_recall': metrics.seg.mr,
}])
display(summary)
print('For paper counting, prioritize mask recall while checking false positives visually.')

test_images = list_images(OUTPUT_DATASET / 'test' / 'images')
preview_sources = random.sample(test_images, min(16, len(test_images)))
predictions = best_model.predict(source=[str(path) for path in preview_sources], imgsz=960, conf=0.10, iou=0.60, max_det=30, device=0, verbose=False)
fig, axes = plt.subplots(4, 4, figsize=(18, 18))
for axis in axes.flat:
    axis.axis('off')
for axis, result in zip(axes.flat, predictions):
    axis.imshow(result.plot()[:, :, ::-1])
    axis.set_title(Path(result.path).name[:40], fontsize=8)
plt.tight_layout()


## 6. Package the checkpoint

Download the `.pt` and metadata from Kaggle Output. Do not replace the current banned-object model until the new segmentation checkpoint passes both project videos.

In [ ]:
FINAL_PT = Path('/kaggle/working/yolo26s_paper_seg_best.pt')
shutil.copy2(BEST_PT, FINAL_PT)
metadata = {
    'task': 'instance_segmentation', 'base_model': MODEL_NAME,
    'class_names': ['paper'], 'train_image_size': 768,
    'recommended_roi_inference_size': 960, 'recommended_confidence_start': 0.10,
    'box_metrics': {'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map), 'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr)},
    'mask_metrics': {'map50': float(metrics.seg.map50), 'map50_95': float(metrics.seg.map), 'precision': float(metrics.seg.mp), 'recall': float(metrics.seg.mr)},
}
METADATA_JSON = Path('/kaggle/working/yolo26s_paper_seg_metadata.json')
METADATA_JSON.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
ARCHIVE_ZIP = Path(shutil.make_archive('/kaggle/working/yolo26s_paper_seg_training_run', 'zip', root_dir=RUN_DIR))
display(FileLink(str(FINAL_PT)))
display(FileLink(str(METADATA_JSON)))
display(FileLink(str(ARCHIVE_ZIP)))
